In [0]:
#Load Silver Table
orders_silver = spark.table("default.orders_silver")
orders_silver.show()

+----------+-----------+--------+--------+-----+-------------------+-------------+-----------------+-------+------------+-----------+
|product_id|customer_id|order_id|quantity|price|          timestamp|customer_name|            email|country|product_name|   category|
+----------+-----------+--------+--------+-----+-------------------+-------------+-----------------+-------+------------+-----------+
|      2002|        102|       2|       1| 20.0|2024-01-01 11:00:00|          Bob|    bob@email.com|Germany|       Phone|Electronics|
|      2001|        101|       4|       1| 15.5|2024-01-02 09:00:00|        Alice|  alice@email.com| France|      Laptop|Electronics|
|      2002|        103|       5|       3| 20.0|2024-01-02 10:00:00|      Charlie|charlie@email.com|Belgium|       Phone|Electronics|
|      2001|        101|       1|       2| 15.5|2024-01-01 10:00:00|        Alice|  alice@email.com| France|      Laptop|Electronics|
+----------+-----------+--------+--------+-----+--------------

In [0]:
#Total Revenue per Product
from pyspark.sql.functions import sum, col

sales_gold = (
    orders_silver
    .withColumn("revenue", col("quantity") * col("price"))
    .groupBy("product_id", "product_name")
    .agg(sum("revenue").alias("total_revenue"))
)

sales_gold.show()

+----------+------------+-------------+
|product_id|product_name|total_revenue|
+----------+------------+-------------+
|      2002|       Phone|         80.0|
|      2001|      Laptop|         46.5|
+----------+------------+-------------+



In [0]:
#Save as Gold Table
sales_gold.write.format("delta").mode("overwrite").saveAsTable("default.sales_gold")

In [0]:
spark.table("default.sales_gold").show()

+----------+------------+-------------+
|product_id|product_name|total_revenue|
+----------+------------+-------------+
|      2001|      Laptop|         46.5|
|      2002|       Phone|         80.0|
+----------+------------+-------------+



In [0]:
from pyspark.sql.functions import to_date

daily_sales = (
    orders_silver
    .withColumn("revenue", col("quantity") * col("price"))
    .withColumn("order_date", to_date("timestamp"))
    .groupBy("order_date")
    .agg(sum("revenue").alias("daily_revenue"))
)

daily_sales.show()

+----------+-------------+
|order_date|daily_revenue|
+----------+-------------+
|2024-01-01|         51.0|
|2024-01-02|         75.5|
+----------+-------------+



In [0]:
top_customers = (
    orders_silver
    .withColumn("revenue", col("quantity") * col("price"))
    .groupBy("customer_id", "customer_name")
    .agg(sum("revenue").alias("total_spent"))
    .orderBy(col("total_spent").desc())
)

top_customers.show()

+-----------+-------------+-----------+
|customer_id|customer_name|total_spent|
+-----------+-------------+-----------+
|        103|      Charlie|       60.0|
|        101|        Alice|       46.5|
|        102|          Bob|       20.0|
+-----------+-------------+-----------+



In [0]:
%sql
SELECT * FROM default.sales_gold;

product_id,product_name,total_revenue
2001,Laptop,46.5
2002,Phone,80.0
